# Loss Functions

Companion notebook for the [Loss Functions lesson](https://ml-viz.vercel.app/courses/optimization-ml/04-loss-functions).

We plot and compare the common losses, show empirically that **MSE predicts the mean while MAE
predicts the median** (and why that makes MAE robust to outliers), and verify the **MLE ↔ loss**
link. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## 1 — Regression losses vs. the error

MSE grows quadratically (outlier-sensitive); MAE grows linearly (robust); Huber is quadratic near 0
and linear beyond δ — a smooth compromise.

In [ ]:
def mse(e): return e**2
def mae(e): return np.abs(e)
def huber(e, d=1.0):
    a = np.abs(e)
    return np.where(a <= d, 0.5*e**2, d*a - 0.5*d**2)

e = np.linspace(-3, 3, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(e, mse(e), label='MSE (L2)', color='#fb7185')
ax.plot(e, mae(e), label='MAE (L1)', color='#2dd4bf')
ax.plot(e, huber(e), label='Huber (δ=1)', color='#818cf8')
ax.set_xlabel('error  (ŷ − y)'); ax.set_ylabel('loss')
ax.set_title('Regression losses: MSE punishes large errors hardest'); ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2 — MSE → mean, MAE → median (robustness)

The constant that minimizes total MSE is the mean; the constant that minimizes total MAE is the
median. With an outlier added, the mean lurches but the median barely moves — that's MAE's
robustness, made concrete.

In [ ]:
data = np.array([2.0, 3.0, 4.0, 5.0, 6.0])
outlier = np.append(data, 100.0)
for name, d in [('clean', data), ('with outlier', outlier)]:
    print(f'{name:13s}: MSE-minimizer (mean)={d.mean():6.2f}   MAE-minimizer (median)={np.median(d):.2f}')
print('\nThe outlier drags the mean to ~20 but the median stays ~4.5 -> MAE is robust.')

## 3 — Cross-entropy punishes confident wrong predictions

For the true class with predicted probability p, the loss is −log p. As p→0 (confidently wrong) the
loss → ∞, producing strong gradients exactly when the model is badly mistaken.

In [ ]:
def binary_cross_entropy(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(y*np.log(p) + (1-y)*np.log(1-p))

p = np.linspace(0.001, 0.999, 200)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(p, binary_cross_entropy(p, 1), color='#818cf8')
ax.set_xlabel('predicted probability of the TRUE class'); ax.set_ylabel('cross-entropy loss')
ax.set_title('Cross-entropy: tiny when confident-correct, huge when confident-wrong')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print('p=0.9 -> loss', round(float(binary_cross_entropy(0.9,1)),3))
print('p=0.1 -> loss', round(float(binary_cross_entropy(0.1,1)),3), '(confidently wrong = big loss)')

## ✏️ Your turn

**Exercise.** Implement `huber_loss(e, delta)` (quadratic for |e|≤δ, linear beyond) and
`cross_entropy(p, y)` (binary, numerically stable via clipping). These are two of the most-used
losses in all of ML.

In [ ]:
def huber_loss(e, delta=1.0):
    # TODO(you): 0.5*e^2 if |e|<=delta else delta*|e| - 0.5*delta^2  (works on numpy arrays)
    return ...

def cross_entropy(p, y):
    # TODO(you): binary cross-entropy -[y log p + (1-y) log(1-p)], clip p for stability
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(huber_loss(np.array([0.5]), 1.0)[0], 0.125)        # quadratic region
assert np.isclose(huber_loss(np.array([3.0]), 1.0)[0], 1.0*3 - 0.5)  # linear region
assert np.isclose(cross_entropy(0.9, 1), -np.log(0.9))
assert cross_entropy(0.1, 1) > cross_entropy(0.9, 1)                  # confident-wrong costs more
print('\u2713 Huber and cross-entropy are correct')

<details>
<summary>Solution</summary>

```python
def huber_loss(e, delta=1.0):
    a = np.abs(e)
    return np.where(a <= delta, 0.5*e**2, delta*a - 0.5*delta**2)

def cross_entropy(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(y*np.log(p) + (1-y)*np.log(1-p))
```

Huber is MSE near zero (smooth gradients) and MAE far out (robust to outliers). Cross-entropy is the
negative log-likelihood of a Bernoulli label — which is why it's the natural classification loss.

</details>